# 01 — Generate and validate SA metric branches

Generates all **24** Structural Analysis branches (6 metrics × 4 types: Bug, BugFX, TCC, CC)
under `build/`, then runs type-specific assert validators.

Library code lives in `lib/sa_generator.py`, `lib/sa_metrics.py`, and `lib/sa_validate.py`.

## Parameters

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "lib" / "sa_generator.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from lib import sa_metrics
from lib.sa_metrics import FIXED_BRANCH_TYPES, METRIC_ABBREVS, branch_name, branches_for_metrics
from lib.sa_generator import generate_all_branches, create_git_branches, push_branches
from lib.sa_validate import BranchValidationError, validate_build

VERSION = "2.6"
LANGUAGE = "python"
METRICS = sa_metrics.ALL_METRICS
BUILD_DIR = "build"
DO_GIT = True
DO_PUSH = True

print("Repo root:", ROOT)
print("Metrics arg:", METRICS)
print("All abbrevs:", ", ".join(METRIC_ABBREVS))
print("Types:", ", ".join(FIXED_BRANCH_TYPES))

Repo root: d:\Metric_evaluation
Metrics arg: all
All abbrevs: EPI, DOV, LSV, TLCC, TDI, QRA
Types: Bug, BugFX, TCC, CC


## 1. Generate branches

In [2]:
generated = generate_all_branches(
    metrics=METRICS,
    version=VERSION,
    language=LANGUAGE,
    build_dir=BUILD_DIR,
    repo_root=str(ROOT),
)
build_root = ROOT / BUILD_DIR
print(f"Generated {len(generated)} branches under {build_root}")

Generated 24 branches under d:\Metric_evaluation\build


## 2. Validate (Bug / BugFX / TCC / CC asserts)

In [3]:
try:
    results = validate_build(
        METRICS, VERSION, LANGUAGE, BUILD_DIR, str(ROOT))
except BranchValidationError as exc:
    raise

for name, bt in results:
    print(f"OK  {name} ({bt})")
print(f"\nAll {len(results)} branches passed validation.")

OK  SA_EPI_Bug_2.6 (Bug)
OK  SA_EPI_BugFX_2.6 (BugFX)
OK  SA_EPI_TCC_2.6 (TCC)
OK  SA_EPI_CC_2.6 (CC)
OK  SA_DOV_Bug_2.6 (Bug)
OK  SA_DOV_BugFX_2.6 (BugFX)
OK  SA_DOV_TCC_2.6 (TCC)
OK  SA_DOV_CC_2.6 (CC)
OK  SA_LSV_Bug_2.6 (Bug)
OK  SA_LSV_BugFX_2.6 (BugFX)
OK  SA_LSV_TCC_2.6 (TCC)
OK  SA_LSV_CC_2.6 (CC)
OK  SA_TLCC_Bug_2.6 (Bug)
OK  SA_TLCC_BugFX_2.6 (BugFX)
OK  SA_TLCC_TCC_2.6 (TCC)
OK  SA_TLCC_CC_2.6 (CC)
OK  SA_TDI_Bug_2.6 (Bug)
OK  SA_TDI_BugFX_2.6 (BugFX)
OK  SA_TDI_TCC_2.6 (TCC)
OK  SA_TDI_CC_2.6 (CC)
OK  SA_QRA_Bug_2.6 (Bug)
OK  SA_QRA_BugFX_2.6 (BugFX)
OK  SA_QRA_TCC_2.6 (TCC)
OK  SA_QRA_CC_2.6 (CC)

All 24 branches passed validation.


## Summary — LOC per branch (sa/ + tests/ + main.py + sa/config.py)

In [4]:
def count_branch_loc(path):
    total = 0
    for dirpath, _, files in os.walk(path):
        for fn in files:
            if not fn.endswith(".py"):
                continue
            rel = os.path.relpath(os.path.join(dirpath, fn), path).replace("\\", "/")
            if rel == "main.py" or rel.startswith("sa/") or rel.startswith("tests/"):
                with open(os.path.join(dirpath, fn), encoding="utf-8") as fh:
                    total += sum(1 for _ in fh)
    return total

import os

rows = []
for bname in generated:
    loc = count_branch_loc(build_root / bname)
    rows.append((bname, loc, "OK" if loc >= 1000 else "FAIL"))

print(f"{'Branch':<24} {'LOC':>6}  Status")
print("-" * 40)
for bname, loc, status in rows:
    print(f"{bname:<24} {loc:>6}  {status}")
print(f"\n{sum(1 for _, loc, s in rows if s == 'OK')}/{len(rows)} branches >= 1000 LOC")

Branch                      LOC  Status
----------------------------------------
SA_EPI_Bug_2.6             1071  OK
SA_EPI_BugFX_2.6           1109  OK
SA_EPI_TCC_2.6             1031  OK
SA_EPI_CC_2.6              1085  OK
SA_DOV_Bug_2.6             1000  OK
SA_DOV_BugFX_2.6           1330  OK
SA_DOV_TCC_2.6             1410  OK
SA_DOV_CC_2.6              1297  OK
SA_LSV_Bug_2.6             1094  OK
SA_LSV_BugFX_2.6           1130  OK
SA_LSV_TCC_2.6             1002  OK
SA_LSV_CC_2.6              1098  OK
SA_TLCC_Bug_2.6            1018  OK
SA_TLCC_BugFX_2.6          1056  OK
SA_TLCC_TCC_2.6            1158  OK
SA_TLCC_CC_2.6             1124  OK
SA_TDI_Bug_2.6             1186  OK
SA_TDI_BugFX_2.6           1234  OK
SA_TDI_TCC_2.6             1090  OK
SA_TDI_CC_2.6              1162  OK
SA_QRA_Bug_2.6             1268  OK
SA_QRA_BugFX_2.6           1272  OK
SA_QRA_TCC_2.6             1312  OK
SA_QRA_CC_2.6              1272  OK

24/24 branches >= 1000 LOC


## 3. Optional — create git branches

In [5]:
if DO_GIT:
    create_git_branches(METRICS, VERSION, LANGUAGE, str(ROOT), BUILD_DIR)
    if DO_PUSH:
        push_branches(generated, str(ROOT))
        print("Pushed branches to origin")
    else:
        print("Git branches created locally")
else:
    print("Skipped git branch creation (DO_GIT=False)")

+ git checkout -B SA_EPI_Bug_2.6 main
+ git add -A sa tests README.md main.py requirements.txt versions.txt
+ git commit -m Add metric-scoped SA SA_EPI_Bug_2.6 codebase
+ git checkout -B SA_EPI_BugFX_2.6 main
+ git add -A sa tests README.md main.py requirements.txt versions.txt
+ git commit -m Add metric-scoped SA SA_EPI_BugFX_2.6 codebase
+ git checkout -B SA_EPI_TCC_2.6 main
+ git add -A sa tests README.md main.py requirements.txt versions.txt .coveragerc .testmondata.ini pytest.ini setup.cfg
+ git commit -m Add metric-scoped SA SA_EPI_TCC_2.6 codebase
+ git checkout -B SA_EPI_CC_2.6 main
+ git add -A sa tests README.md main.py requirements.txt versions.txt
+ git commit -m Add metric-scoped SA SA_EPI_CC_2.6 codebase
+ git checkout -B SA_DOV_Bug_2.6 main
+ git add -A sa tests README.md main.py requirements.txt versions.txt
+ git commit -m Add metric-scoped SA SA_DOV_Bug_2.6 codebase
+ git checkout -B SA_DOV_BugFX_2.6 main
+ git add -A sa tests README.md main.py requirements.txt versio